# Preparación de la base para la segmentación de comercios

### Objetivo

Construir una base consolidada que contenga una sola fila por comercio, integrando la información de sus terminales, ventas y comportamiento transaccional durante los seis meses disponibles.

Para lograrlo se realizarán las siguientes actividades:

* Consolidar las terminales asociadas a cada comercio.
* Sumar las ventas registradas en sus diferentes terminales.
* Resumir su actividad transaccional de los seis meses.
* Construir variables como frecuencia, meses activos, recencia y tendencia.
* Integrar las bases de comercios y transacciones.
* Validar que el resultado tenga una sola fila por Id_comercio.
* Exportar la base final que será utilizada posteriormente en el modelo de segmentación.



In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

#config
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

sns.set_theme(style="whitegrid", palette="muted")

### 1. Cargue de datos


In [2]:
archivo = "../data/raw/DataTransacciones_PruebasIngreso.xlsx"

comercios = pd.read_excel(
    archivo,
    sheet_name="comercios",
    dtype={"Id_comercio": "string", "Id_terminal": "string"}
)

transacciones = pd.read_excel(
    archivo,
    sheet_name="transacciones",
    dtype={"Id_comercio": "string"}
)

In [3]:
display(comercios.head())
display(transacciones.head())

,Id_comercio,Fecha vinculación datafono,Fecha de creacion comercio,Categoria comercio,Sub Categoria comercio,Id_terminal,Ciudad,Fecha 1ra_transaccion,Monto ventas Acumulado,Monto ventas ultimo mes
0,003QDVFJCJ,"mar. 26, 2021 12:40pm","mar. 26, 2021 12:17pm",Comercio detallista,Ropa / Accesorios,01299,Bogotá D.C.,"mar. 26, 2021 12:42pm",6542000,1240000
1,008LJH6YHN,"feb. 10, 2021 3:31pm","feb. 10, 2021 3:15pm",Belleza y cuidado personal,Spa,00591,Bogotá D.C.,"feb. 10, 2021 3:34pm",4380000,1102000
2,00B3DGV62G,"mar. 3, 2021 3:26pm","mar. 3, 2021 3:09pm",Comercio detallista,Auto partes,02130,Bogotá D.C.,"mar. 3, 2021 3:30pm",11892500,1297000
3,00ILW1BB3Y,"mar. 24, 2021 12:45pm","mar. 24, 2021 12:33pm",Comercio detallista,Tienda naturista,00662,Bogotá D.C.,"mar. 24, 2021 12:51pm",372500,74500
4,01AK7UX4UU,"mar. 30, 2021 1:50pm","mar. 30, 2021 1:40pm",Comercio detallista,Tecnología,01784,Mosquera,"mar. 30, 2021 1:52pm",165500,0


,Id_comercio,mes-anio transaccion,cantidad transacciones
0,003QDVFJCJ,3-2021,5
1,003QDVFJCJ,4-2021,9
2,003QDVFJCJ,5-2021,16
3,008LJH6YHN,2-2021,2
4,008LJH6YHN,3-2021,5


#### 2. Preparación de fechas

##### tabla transacciones

In [4]:
# Convertir la columna de periodo a tipo fecha
transacciones["fecha_mes"] = pd.to_datetime(
    transacciones["mes-anio transaccion"],
    format="%m-%Y"
)

# Identificar los meses disponibles
meses_disponibles = (
    transacciones["fecha_mes"]
    .drop_duplicates()
    .sort_values()
)

fecha_inicial = meses_disponibles.min()
ultimo_mes = meses_disponibles.max()

print("Fecha inicial:", fecha_inicial)
print("Fecha final:", ultimo_mes)
print("Cantidad de meses:", meses_disponibles.nunique())

display(
    meses_disponibles
    .dt.strftime("%Y-%m")
    .to_frame(name="Mes_disponible")
)

Fecha inicial: 2020-12-01 00:00:00
Fecha final: 2021-05-01 00:00:00
Cantidad de meses: 6


,Mes_disponible
15,2020-12
56,2021-01
3,2021-02
0,2021-03
1,2021-04
2,2021-05


In [5]:
ruta_transacciones_mensuales = Path(
    "../data/processed/transacciones_mensuales.csv"
)

transacciones.to_csv(
    ruta_transacciones_mensuales,
    index=False
)

print("Archivo guardado en:", ruta_transacciones_mensuales)

Archivo guardado en: ../data/processed/transacciones_mensuales.csv


##### tabla comercios

In [6]:
# Meses en español
meses = {
    "ene.": "01",
    "feb.": "02",
    "mar.": "03",
    "abr.": "04",
    "may.": "05",
    "jun.": "06",
    "jul.": "07",
    "ago.": "08",
    "sept.": "09",
    "sep.": "09",
    "oct.": "10",
    "nov.": "11",
    "dic.": "12"
}

In [7]:
def convertir_fecha(columna):
    fecha = columna.copy()

    for mes, numero in meses.items():
        fecha = fecha.str.replace(
            mes,
            numero,
            regex=False
        )

    return pd.to_datetime(
        fecha,
        format="%m %d, %Y %I:%M%p",
        errors="coerce"
    )

In [8]:
comercios["fecha_vinculacion"] = convertir_fecha(
    comercios["Fecha vinculación datafono"]
)

comercios["fecha_creacion"] = convertir_fecha(
    comercios["Fecha de creacion comercio"]
)

comercios["fecha_primera_transaccion"] = convertir_fecha(
    comercios["Fecha 1ra_transaccion"]
)

In [9]:
print(
    "Fechas de vinculación vacías:",
    comercios["fecha_vinculacion"].isna().sum()
)

print(
    "Fechas de creación vacías:",
    comercios["fecha_creacion"].isna().sum()
)

print(
    "Fechas de primera transacción vacías:",
    comercios["fecha_primera_transaccion"].isna().sum()
)

display(
    comercios[
        [
            "fecha_vinculacion",
            "fecha_creacion",
            "fecha_primera_transaccion"
        ]
    ].head()
)

Fechas de vinculación vacías: 0
Fechas de creación vacías: 0
Fechas de primera transacción vacías: 21


,fecha_vinculacion,fecha_creacion,fecha_primera_transaccion
0,2021-03-26 12:40:00,2021-03-26 12:17:00,2021-03-26 12:42:00
1,2021-02-10 15:31:00,2021-02-10 15:15:00,2021-02-10 15:34:00
2,2021-03-03 15:26:00,2021-03-03 15:09:00,2021-03-03 15:30:00
3,2021-03-24 12:45:00,2021-03-24 12:33:00,2021-03-24 12:51:00
4,2021-03-30 13:50:00,2021-03-30 13:40:00,2021-03-30 13:52:00


#### 3. Consolidación de comercios y terminales

In [10]:
# Consolidar la información por comercio
comercios_consolidados = (
    comercios.groupby("Id_comercio")
    .agg({
        "Id_terminal": "nunique",
        "Categoria comercio": "first",
        "Sub Categoria comercio": "first",
        "Ciudad": "first",
        "fecha_vinculacion": "min",
        "fecha_creacion": "min",
        "fecha_primera_transaccion": "min",
        "Monto ventas Acumulado": "sum",
        "Monto ventas ultimo mes": "sum"
    })
    .reset_index()
)

comercios_consolidados.columns = [
    "Id_comercio",
    "Numero_terminales",
    "Categoria",
    "Subcategoria",
    "Ciudad",
    "Fecha_primera_vinculacion",
    "Fecha_creacion_comercio",
    "Fecha_primera_transaccion",
    "Ventas_acumuladas",
    "Ventas_ultimo_mes_reportado"
]

comercios_consolidados.head()
display(comercios_consolidados.head())

print("Filas:", len(comercios_consolidados))
print(
    "Comercios únicos:",
    comercios_consolidados["Id_comercio"].nunique()
)
print(
    "Comercios duplicados:",
    comercios_consolidados["Id_comercio"].duplicated().sum()
)

,Id_comercio,Numero_terminales,Categoria,Subcategoria,Ciudad,Fecha_primera_vinculacion,Fecha_creacion_comercio,Fecha_primera_transaccion,Ventas_acumuladas,Ventas_ultimo_mes_reportado
0,003QDVFJCJ,1,Comercio detallista,Ropa / Accesorios,Bogotá D.C.,2021-03-26 12:40:00,2021-03-26 12:17:00,2021-03-26 12:42:00,6542000,1240000
1,008LJH6YHN,1,Belleza y cuidado personal,Spa,Bogotá D.C.,2021-02-10 15:31:00,2021-02-10 15:15:00,2021-02-10 15:34:00,4380000,1102000
2,00B3DGV62G,1,Comercio detallista,Auto partes,Bogotá D.C.,2021-03-03 15:26:00,2021-03-03 15:09:00,2021-03-03 15:30:00,11892500,1297000
3,00ILW1BB3Y,1,Comercio detallista,Tienda naturista,Bogotá D.C.,2021-03-24 12:45:00,2021-03-24 12:33:00,2021-03-24 12:51:00,372500,74500
4,01AK7UX4UU,1,Comercio detallista,Tecnología,Mosquera,2021-03-30 13:50:00,2021-03-30 13:40:00,2021-03-30 13:52:00,165500,0


Filas: 3015
Comercios únicos: 3015
Comercios duplicados: 0


In [11]:
ruta_comercios = Path(
    "../data/interim/comercios_consolidados.csv"
)

comercios_consolidados.to_csv(
    ruta_comercios,
    index=False
)

print("Archivo guardado en:", ruta_comercios)

Archivo guardado en: ../data/interim/comercios_consolidados.csv


#### 4. Construcción de la matriz mensual

In [12]:
transacciones.head()

,Id_comercio,mes-anio transaccion,cantidad transacciones,fecha_mes
0,003QDVFJCJ,3-2021,5,2021-03-01
1,003QDVFJCJ,4-2021,9,2021-04-01
2,003QDVFJCJ,5-2021,16,2021-05-01
3,008LJH6YHN,2-2021,2,2021-02-01
4,008LJH6YHN,3-2021,5,2021-03-01


In [13]:
matriz_mensual = transacciones.pivot_table(
    index="Id_comercio",
    columns="fecha_mes",
    values="cantidad transacciones",
    aggfunc="sum",
    fill_value=0
)
matriz_mensual = matriz_mensual.reindex(
    columns=meses_disponibles,
    fill_value=0
)

matriz_mensual.columns = [
    f"Tx_{fecha.strftime('%Y_%m')}"
    for fecha in matriz_mensual.columns
]

matriz_mensual = matriz_mensual.reset_index()

display(matriz_mensual.head())

,Id_comercio,Tx_2020_12,Tx_2021_01,Tx_2021_02,Tx_2021_03,Tx_2021_04,Tx_2021_05
0,003QDVFJCJ,0,0,0,5,9,16
1,008LJH6YHN,0,0,2,5,1,2
2,00B3DGV62G,0,0,0,22,22,21
3,00ILW1BB3Y,0,0,0,1,2,5
4,01AK7UX4UU,0,0,0,1,3,0


#### 5. Construcción de variables transaccionales

In [14]:
columnas_meses = [
    columna
    for columna in matriz_mensual.columns
    if columna.startswith("Tx_")
]

transacciones_consolidadas = matriz_mensual.copy()

In [15]:
# Total de transacciones en los seis meses
transacciones_consolidadas["Total_transacciones_6m"] = (
    transacciones_consolidadas[columnas_meses]
    .sum(axis=1)
)

# Cantidad de meses con transacciones
transacciones_consolidadas["Meses_activos"] = (
    transacciones_consolidadas[columnas_meses]
    .gt(0)
    .sum(axis=1)
)

In [16]:
ultimo_mes_por_comercio = (
    transacciones.groupby("Id_comercio")["fecha_mes"]
    .max()
    .reset_index()
)

ultimo_mes_por_comercio.columns = [
    "Id_comercio",
    "Ultimo_mes_activo"
]

transacciones_consolidadas = (
    transacciones_consolidadas.merge(
        ultimo_mes_por_comercio,
        on="Id_comercio",
        how="left",
        validate="one_to_one"
    )
)

In [17]:
transacciones_consolidadas

,Id_comercio,Tx_2020_12,Tx_2021_01,Tx_2021_02,Tx_2021_03,Tx_2021_04,Tx_2021_05,Total_transacciones_6m,Meses_activos,Ultimo_mes_activo
0,003QDVFJCJ,0,0,0,5,9,16,30,3,2021-05-01
1,008LJH6YHN,0,0,2,5,1,2,10,4,2021-05-01
2,00B3DGV62G,0,0,0,22,22,21,65,3,2021-05-01
3,00ILW1BB3Y,0,0,0,1,2,5,8,3,2021-05-01
4,01AK7UX4UU,0,0,0,1,3,0,4,2,2021-04-01
...,...,...,...,...,...,...,...,...,...,...
3010,ZYU8UUM32R,0,0,0,4,5,9,18,3,2021-05-01
3011,ZYV1ZCZ7IM,0,0,0,2,5,4,11,3,2021-05-01
3012,ZZJ3KWF9VC,0,0,0,2,0,3,5,2,2021-05-01
3013,ZZMG2NE0Q6,0,0,0,2,1,1,4,3,2021-05-01


In [18]:
#Recencia meses de actividad hacía atrás desde el último mes reportado
transacciones_consolidadas["Recencia_meses"] = (
    (ultimo_mes.year
     - transacciones_consolidadas["Ultimo_mes_activo"].dt.year) * 12
    + (ultimo_mes.month
       - transacciones_consolidadas["Ultimo_mes_activo"].dt.month)
)

transacciones_consolidadas 

,Id_comercio,Tx_2020_12,Tx_2021_01,Tx_2021_02,Tx_2021_03,Tx_2021_04,Tx_2021_05,Total_transacciones_6m,Meses_activos,Ultimo_mes_activo,Recencia_meses
0,003QDVFJCJ,0,0,0,5,9,16,30,3,2021-05-01,0
1,008LJH6YHN,0,0,2,5,1,2,10,4,2021-05-01,0
2,00B3DGV62G,0,0,0,22,22,21,65,3,2021-05-01,0
3,00ILW1BB3Y,0,0,0,1,2,5,8,3,2021-05-01,0
4,01AK7UX4UU,0,0,0,1,3,0,4,2,2021-04-01,1
...,...,...,...,...,...,...,...,...,...,...,...
3010,ZYU8UUM32R,0,0,0,4,5,9,18,3,2021-05-01,0
3011,ZYV1ZCZ7IM,0,0,0,2,5,4,11,3,2021-05-01,0
3012,ZZJ3KWF9VC,0,0,0,2,0,3,5,2,2021-05-01,0
3013,ZZMG2NE0Q6,0,0,0,2,1,1,4,3,2021-05-01,0


In [19]:
transacciones_consolidadas["Transacciones_primeros_3m"] = (
    transacciones_consolidadas[columnas_meses[:3]]
    .sum(axis=1)
)

transacciones_consolidadas["Transacciones_ultimos_3m"] = (
    transacciones_consolidadas[columnas_meses[-3:]]
    .sum(axis=1)
)

# Calcular la tendencia de transacciones comparada entre los primeros 3 meses y los últimos 3 meses
transacciones_consolidadas["Tendencia_transacciones"] = (
    transacciones_consolidadas["Transacciones_ultimos_3m"]
    - transacciones_consolidadas["Transacciones_primeros_3m"]
)

transacciones_consolidadas

,Id_comercio,Tx_2020_12,Tx_2021_01,Tx_2021_02,Tx_2021_03,Tx_2021_04,Tx_2021_05,Total_transacciones_6m,Meses_activos,Ultimo_mes_activo,Recencia_meses,Transacciones_primeros_3m,Transacciones_ultimos_3m,Tendencia_transacciones
0,003QDVFJCJ,0,0,0,5,9,16,30,3,2021-05-01,0,0,30,30
1,008LJH6YHN,0,0,2,5,1,2,10,4,2021-05-01,0,2,8,6
2,00B3DGV62G,0,0,0,22,22,21,65,3,2021-05-01,0,0,65,65
3,00ILW1BB3Y,0,0,0,1,2,5,8,3,2021-05-01,0,0,8,8
4,01AK7UX4UU,0,0,0,1,3,0,4,2,2021-04-01,1,0,4,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3010,ZYU8UUM32R,0,0,0,4,5,9,18,3,2021-05-01,0,0,18,18
3011,ZYV1ZCZ7IM,0,0,0,2,5,4,11,3,2021-05-01,0,0,11,11
3012,ZZJ3KWF9VC,0,0,0,2,0,3,5,2,2021-05-01,0,0,5,5
3013,ZZMG2NE0Q6,0,0,0,2,1,1,4,3,2021-05-01,0,0,4,4


In [20]:
ruta_transacciones = Path(
    "../data/interim/transacciones_consolidadas.csv"
)

transacciones_consolidadas.to_csv(
    ruta_transacciones,
    index=False
)

print("Archivo guardado en:", ruta_transacciones)

Archivo guardado en: ../data/interim/transacciones_consolidadas.csv


#### 6. Unión de comercios y transacciones

In [21]:
# Unir la información comercial y transaccional
base_segmentacion = comercios_consolidados.merge(
    transacciones_consolidadas,
    on="Id_comercio",
    how="left",
    validate="one_to_one"
)

# Validar el resultado
print("Filas:", len(base_segmentacion))
print(
    "Comercios únicos:",
    base_segmentacion["Id_comercio"].nunique()
)
print(
    "Identificadores duplicados:",
    base_segmentacion["Id_comercio"].duplicated().sum()
)
print(
    "Comercios sin información transaccional:",
    base_segmentacion["Total_transacciones_6m"].isna().sum()
)

display(base_segmentacion.head())

Filas: 3015
Comercios únicos: 3015
Identificadores duplicados: 0
Comercios sin información transaccional: 0


,Id_comercio,Numero_terminales,Categoria,Subcategoria,Ciudad,Fecha_primera_vinculacion,Fecha_creacion_comercio,Fecha_primera_transaccion,Ventas_acumuladas,Ventas_ultimo_mes_reportado,Tx_2020_12,Tx_2021_01,Tx_2021_02,Tx_2021_03,Tx_2021_04,Tx_2021_05,Total_transacciones_6m,Meses_activos,Ultimo_mes_activo,Recencia_meses,Transacciones_primeros_3m,Transacciones_ultimos_3m,Tendencia_transacciones
0,003QDVFJCJ,1,Comercio detallista,Ropa / Accesorios,Bogotá D.C.,2021-03-26 12:40:00,2021-03-26 12:17:00,2021-03-26 12:42:00,6542000,1240000,0,0,0,5,9,16,30,3,2021-05-01,0,0,30,30
1,008LJH6YHN,1,Belleza y cuidado personal,Spa,Bogotá D.C.,2021-02-10 15:31:00,2021-02-10 15:15:00,2021-02-10 15:34:00,4380000,1102000,0,0,2,5,1,2,10,4,2021-05-01,0,2,8,6
2,00B3DGV62G,1,Comercio detallista,Auto partes,Bogotá D.C.,2021-03-03 15:26:00,2021-03-03 15:09:00,2021-03-03 15:30:00,11892500,1297000,0,0,0,22,22,21,65,3,2021-05-01,0,0,65,65
3,00ILW1BB3Y,1,Comercio detallista,Tienda naturista,Bogotá D.C.,2021-03-24 12:45:00,2021-03-24 12:33:00,2021-03-24 12:51:00,372500,74500,0,0,0,1,2,5,8,3,2021-05-01,0,0,8,8
4,01AK7UX4UU,1,Comercio detallista,Tecnología,Mosquera,2021-03-30 13:50:00,2021-03-30 13:40:00,2021-03-30 13:52:00,165500,0,0,0,0,1,3,0,4,2,2021-04-01,1,0,4,4


#### 7. Variables e indicadores de negocio

In [23]:
# Comisión obtenida por la plataforma
base_segmentacion["Comision_acumulada"] = (
    base_segmentacion["Ventas_acumuladas"] * 0.05
)

base_segmentacion["Comision_ultimo_mes_reportado"] = (
    base_segmentacion["Ventas_ultimo_mes_reportado"] * 0.05
)

# Valor promedio vendido por transacción
base_segmentacion["Venta_promedio_por_transaccion"] = (
    base_segmentacion["Ventas_acumuladas"]
    / base_segmentacion["Total_transacciones_6m"]
)

# Ventas promedio por terminal
base_segmentacion["Ventas_por_terminal"] = (
    base_segmentacion["Ventas_acumuladas"]
    / base_segmentacion["Numero_terminales"]
)

columnas_indicadores = [
    "Id_comercio",
    "Ventas_acumuladas",
    "Total_transacciones_6m",
    "Numero_terminales",
    "Comision_acumulada",
    "Comision_ultimo_mes_reportado",
    "Venta_promedio_por_transaccion",
    "Ventas_por_terminal"
]

display(base_segmentacion[columnas_indicadores].head())

,Id_comercio,Ventas_acumuladas,Total_transacciones_6m,Numero_terminales,Comision_acumulada,Comision_ultimo_mes_reportado,Venta_promedio_por_transaccion,Ventas_por_terminal
0,003QDVFJCJ,6542000,30,1,"327,100.00","62,000.00","218,066.67","6,542,000.00"
1,008LJH6YHN,4380000,10,1,"219,000.00","55,100.00","438,000.00","4,380,000.00"
2,00B3DGV62G,11892500,65,1,"594,625.00","64,850.00","182,961.54","11,892,500.00"
3,00ILW1BB3Y,372500,8,1,"18,625.00","3,725.00","46,562.50","372,500.00"
4,01AK7UX4UU,165500,4,1,"8,275.00",0.00,"41,375.00","165,500.00"


In [26]:
# Validar la granularidad de la base final
print("Filas:", len(base_segmentacion))
print(
    "Comercios únicos:",
    base_segmentacion["Id_comercio"].nunique()
)
print(
    "Identificadores duplicados:",
    base_segmentacion["Id_comercio"].duplicated().sum()
)
print(
    "Total de terminales:",
    base_segmentacion["Numero_terminales"].sum()
)
print(
    "Total de transacciones:",
    base_segmentacion["Total_transacciones_6m"].sum()
)
print(
    "Valores nulos totales:",
    base_segmentacion.isna().sum().sum()
)

Filas: 3015
Comercios únicos: 3015
Identificadores duplicados: 0
Total de terminales: 3071
Total de transacciones: 68152
Valores nulos totales: 11


In [27]:
# Guardar la base que se utilizará en la segmentación
ruta_base_modelo = Path(
    "../data/processed/base_modelo_segmentacion.csv"
)

base_segmentacion.to_csv(
    ruta_base_modelo,
    index=False,
    date_format="%Y-%m-%d"
)

print("Archivo guardado en:", ruta_base_modelo)

Archivo guardado en: ../data/processed/base_modelo_segmentacion.csv
